In [1]:
import pandas as pd
import geopandas as gpd
import json
import numpy as np

In [2]:
eia_930_ref_subbas = pd.read_excel('../data/EIA930_Reference_Tables.xlsx', sheet_name='BA Subregions')
subba_parent_map = dict(zip(
    eia_930_ref_subbas['BA Subregion Code'],
    eia_930_ref_subbas['BA Code']
))

In [3]:
county_geo = gpd.read_file("../data/shapefiles/US_COUNTY_2022")
ba_subba_geo = gpd.read_file("../data/shapefiles/bas_and_subbas")
ba_subba_geo['iso'] = ba_subba_geo['EIAcode'].map(subba_parent_map)
subba_geo = (
    ba_subba_geo.loc[ba_subba_geo.iso != 'PNM']
    .dropna(subset='iso')
    .copy()
)
subba_geo['zone_area'] = subba_geo.area

In [4]:
def create_county_subba_map(ba, county_geo, subba_geo):
    ba_subba_geo = subba_geo.loc[subba_geo.iso == ba]
    
    county_subba_intersects = gpd.overlay(
        county_geo,
        ba_subba_geo,
        how='intersection',
        keep_geom_type=False
    )
    county_subba_intersects['area'] = (
        (county_subba_intersects.geometry.area / county_subba_intersects['zone_area'])
        .round(3)
    )
    county_subba_codes = (
        county_subba_intersects.loc[county_subba_intersects['area'] > 0]
        .groupby('rb')
        ['EIAcode']
        .apply(list)
    )

    return county_subba_codes

county_subba_maps = {}
for ba in subba_geo.iso.unique().tolist():
    county_subba_maps[ba] = create_county_subba_map(
        ba, county_geo, subba_geo
    )
    print(f"Completed {ba}")

Completed CISO
Completed ISNE
Completed NYIS
Completed ERCO
Completed MISO
Completed SWPP
Completed PJM


In [5]:
for ba, data in county_subba_maps.items():
    with open(f"../config/{ba.lower()}_county_subregion_map.json", 'w') as f:
        json.dump(data.to_dict(), f, indent=4)